In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go
import logging

import utils
import z_utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
loader = utils.get_m2s_loader(128)

# Sparate the data into train and test
train_loader = []
test_loader = []

for i, data in enumerate(loader):
    if i % 10 == 0:
        test_loader.append(data)
    else:
        train_loader.append(data)

test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')
print(test_df.shape)
print(test_df.head())


(1682533, 19)
       LCLid  energy(kWh/hh)_smoothed  energy(kWh/hh)   mean  median     std  \
0  MAC000051                   0.3409           0.247  0.323    0.21  0.3232   
1  MAC000051                   0.3760           0.318  0.323    0.21  0.3232   
2  MAC000051                   0.3652           0.284  0.323    0.21  0.3232   
3  MAC000051                   0.4082           0.306  0.323    0.21  0.3232   
4  MAC000051                   0.4033           0.337  0.323    0.21  0.3232   

   min    max  gradient  kmedoid_clusters  spike_type  spike_magnitude  \
0  0.0  3.462    0.1614                17           0              1.0   
1  0.0  3.462    0.1614                17           0              1.0   
2  0.0  3.462    0.1614                17           0              1.0   
3  0.0  3.462    0.1614                17           0              1.0   
4  0.0  3.462    0.1614                17           0              1.0   

   temperature  windSpeed  humidity  date_sin  date_cos  tim

## GAN

In [3]:
class Generator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, hidden_size):
        super(Generator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude), dim=-1)

        return self.fc(out)

# We will put all the auxiliary information for the Discriminator too, to make it better.
class Discriminator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, fc1_hidden_size, fc1_out, fc2_hidden_size):
        super(Discriminator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )
        
        self.fc1 = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1 + 1, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_out),
            nn.ReLU()
        )

        self.fc2 = nn.Sequential(
            nn.Linear(fc1_out + 1, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, 1)
        )

    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude, energy):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude, energy), dim=-1)
        out = self.fc1(out)
        out = torch.cat((out, energy), dim=-1)
        return self.fc2(out)

In [4]:
best_gen_config_path = 'Config/wgan_g_config.json'
with open(best_gen_config_path) as f:
    best_gen_config = json.load(f)

best_dis_config_path = 'Config/wgan_d_config.json'
with open(best_dis_config_path) as f:
    best_dis_config = json.load(f)

gen = Generator(**best_gen_config).to(device)
dis = Discriminator(**best_dis_config).to(device)

gen.load_state_dict(torch.load('../../Models/WGAN_generator.pt'))
dis.load_state_dict(torch.load('../../Models/WGAN_discriminator.pt'))

gen.eval()
dis.eval()


Discriminator(
  (Embedding_spike): Embedding(5, 3)
  (Embedding_cluster): Embedding(20, 2)
  (time_net): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=8, bias=True)
    (3): ReLU()
  )
  (statistical_net): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (fc1): Sequential(
    (0): Linear(in_features=40, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
    (5): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=5, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1, bias=True)
  )
)

## Single VAE

In [5]:
best_config_path = 'Config/s_vae_config.json'

with open(best_config_path, 'r') as file:
    best_config = json.load(file)

s_VAE = network.m2s_VAE(**best_config).to(device)

s_VAE.load_state_dict(torch.load('../../Models/s_vae.pt'))

s_VAE.eval()

m2s_VAE(
  (lstm_encoder): m2s_Encoder(
    (lstm): LSTM(1, 8, num_layers=2, batch_first=True, bidirectional=True)
    (fc1): Sequential(
      (0): Linear(in_features=16, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (4): ReLU()
    )
    (fc2): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=1024, bias=True)
      (4): ReLU()
    )
  )
  (lstm_decoder): m2s_Decoder(
    (Embedding_spike): Embedding(5, 1)
    (Embedding_cluster): Embedding(20, 2)
    (lstm): LSTM(512, 32, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
    (time_net): Sequential(
      (0): Linear(in_features=4, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=32, bi

## Our Method

In [6]:
medoid_config_file = 'Config/medoid_config.json'
m2s_config_file = 'Config/m2s_config.json'

with open(medoid_config_file, 'r') as f:
    medoid_config = json.load(f)

with open(m2s_config_file, 'r') as f:
    m2s_config = json.load(f)

model_medoid = network.m_VAE(**medoid_config).to(device)
model_medoid.load_state_dict(torch.load('../../Models/medoid_vae_best.pt', map_location=device))

model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s.load_state_dict(torch.load('../../Models/m2s_vae_best.pt', map_location=device))

model_medoid.eval()
model_m2s.eval()

clear_output = True

## Together

### Training Set

In [20]:
eva_training_loader = utils.get_m2s_loader(1)
long_GAN_energy = torch.Tensor().to(device)
long_sVAE_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)
long_synthetic_energy = torch.Tensor().to(device)

# For longer sequences
for i in range(12):
    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader))
    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)  # or torch.long for indexing
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    vae_real = real_energy.to(device)

    GAN_energy = gen(weather, cluster, time, statistical, spike_type, spike_magnitude)
    GAN_energy = GAN_energy.to(device)
    long_GAN_energy = torch.cat((long_GAN_energy, GAN_energy), dim=-1)

    noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)
    synthetic_profile, mu, logvar = s_VAE(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)
    long_sVAE_energy = torch.cat((long_sVAE_energy, synthetic_profile), dim=-1)

    noise2 = torch.randn(real_energy.shape[0], real_energy.shape[1], 64).to(device)
    h = model_medoid.lstm_decoder(noise2, weather, cluster, time)
    latent_space = model_m2s.lstm_encoder(h)
    mu, logvar = latent_space.chunk(2, dim=2)
    z = model_m2s.reparameterize(mu, logvar)
    h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)
    long_synthetic_energy = torch.cat((long_synthetic_energy, h), dim=-1)


    real_energy = real_energy.to(device)
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

long_GAN_energy = long_GAN_energy.flatten().detach().cpu().numpy()
long_sVAE_energy = long_sVAE_energy.flatten().detach().cpu().numpy()
long_synthetic_energy = long_synthetic_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

print('MSE of sVAE energy and long_synthetic_energy:', np.mean((long_sVAE_energy - long_synthetic_energy) ** 2))
# plot the results
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_real_energy, mode='lines', name='Real', line=dict(color='green')))
fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_GAN_energy, mode='lines', name='GAN', line=dict(color='orange', dash='dash')))
fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_sVAE_energy, mode='lines', name='Single VAE', line=dict(color='blue', dash='dash')))
fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_synthetic_energy, mode='lines', name='CC-VAE', line=dict(color='red', dash='dot')))

fig.update_layout(
    font=dict(size=18),          # global font (axes, title, etc.)
    legend=dict(font=dict(size=18)),
)

fig.update_layout(
    width=1000,   # make it less wide (try 600–900)
    height=450   # optional
)

fig.update_layout(
    title="Performance Comparison on Training Set Households",
    xaxis_title="Timestamp",
    yaxis_title="Energy (kWh/hh)"
)



fig.show()

MSE of sVAE energy and long_synthetic_energy: 0.022483846


### Test Set

In [31]:
# Get the test df at Data_processed\Kmedoids_with_Weather_Espike_test.csv
test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')
weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(48 * 3, test_df)

weather = weather.to(device)
cluster = cluster.to(device).int()
time = time.to(device)
statistical = statistical.to(device)
spike_type = spike_type.to(dtype=torch.int32)  # or torch.long for indexing
spike_magnitude = spike_magnitude.to(dtype=torch.float32)
spike_type = spike_type.to(device)
spike_magnitude = spike_magnitude.to(device)
real_energy_vae = torch.Tensor(real_energy).to(device)

GAN_energy = gen(weather, cluster, time, statistical, spike_type, spike_magnitude).cpu().detach().numpy().squeeze(0)

real_energy_vae = real_energy_vae.unsqueeze(0)
noise2 = torch.randn(real_energy_vae.shape[0], real_energy_vae.shape[1], 1).to(device)
synthetic_profile, mu, logvar = s_VAE(noise2, weather, cluster, time, statistical, spike_type, spike_magnitude)
sVAE_energy = synthetic_profile.squeeze(0).detach().cpu().numpy()

h = model_medoid.lstm_decoder(noise, weather, cluster, time)
latent_space = model_m2s.lstm_encoder(h)
mu, logvar = latent_space.chunk(2, dim=2)
z = model_m2s.reparameterize(mu, logvar)
h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)
long_synthetic_energy = h.squeeze(0).detach().cpu().numpy()
synthetic_energy = h.squeeze().detach().cpu().numpy()

gradient = statistical.squeeze().detach().cpu().numpy()[0][-1]
min_energy = statistical.squeeze().detach().cpu().numpy()[0][3]
enhanced_synthetic_energy = utils.enhanced_energy_profile(spike_type, synthetic_energy, gradient, min_energy)
enhanced_synthetic_energy = enhanced_synthetic_energy.reshape(-1, 1)


# real_energy = real_energy.flatten().detach().cpu().numpy()
real_energy = real_energy

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=real_energy[:, 0], mode='lines', name='Real', line=dict(color='green')))
fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=GAN_energy[:, 0], mode='lines', name='GAN', line=dict(color='orange', dash='dash')))
fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=sVAE_energy[:, 0], mode='lines', name='sVAE', line=dict(color='blue', dash='dash')))
fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=enhanced_synthetic_energy[:, 0], mode='lines', name='CC-VAE', line=dict(color='red', dash='dot')))


fig.update_layout(
    font=dict(size=18),          # global font (axes, title, etc.)
    legend=dict(font=dict(size=18)),
)

fig.update_layout(
    title="Performance Comparison on Test Set Household",
    xaxis_title="Timestamp",
    yaxis_title="Energy (kWh/hh)"
)

fig.update_layout(
    width=1000,   # make it less wide (try 600–900)
    height=450   # optional
)
fig.show()
